# Architecture

What is the underlying architecture of Kafi Streams?


## Overview

* [A dependency diagram](#dependency)
  * [The TopologyNode class](#topologynode)
  * [The Streams class](#streams)
* [The relationship between TopologyNode and Streams](#relationship)


---
<a id="dependency"></a>
## A dependency diagram

Here is a dependency diagram:

```mermaid
---
title: Kafi Streams dependency diagram
---
classDiagram
    Streams <|-- TopologyNode

    TopologyNode <|-- pydbsp
    TopologyNode <|-- msgpack
    TopologyNode <|-- cloudpickle
```

The two central classes in Kafi Streams are `TopologyNode` and `Streams`.


<a id="topologynode"></a>
### The TopologyNode class

The `TopologyNode` class is the fluent API on top of [*pydbsp*](https://github.com/brurucy/pydbsp) by Bruno Rucy, and is heavily inspired by the Kafka Streams DSL.

The class is completely abstracted away from Kafka. It does not know anything about Kafka. It just receives inputs, processes them relationally using pydbsp, and returns the outputs. This is why it can serve also as a test harness similar to the "TopologyTestDriver" in Kafka Streams.

Stream processing in pydbsp is done in-memory. There are no state stores (even though pydbsp does have a pluggable storage abstraction - but Kafi Streams does not yet take advantage of it). Purely in-memory.

These are the dependencies of the TopologyNode class, from bottom to top.

#### pydbsp

The by far most important building block is pydbsp by Bruno Rucy. It is the heart of Kafi Streams. It is the actual stream processing engine. Information about the integration of 

#### msgpack

In pydbsp, the fundamental data type is the *ZSet*. ZSets are implemented as dictionaries in pydbsp, where the keys are rows and the values are weights (=integers), e.g.:
```python
{"row_1": 1, "row_2": 0, "row_3": -1}
```

As Kafi Streams is typically used on top of Kafka where the payloads are encoded in JSON (and Kafi converts the JSONs into Python dictionaries automatically), I needed a fast way to serialize these dictionaries into a hashable form and deserialize them back to dictionaries.

This is the task of msgpack.

#### cloudpickle

cloudpickle is used for serializing/deserializing the global state of the topology (technically, the state of the pydbsp `evaluator`) since the built-in Python pickler is unable to serialize it.


<a id="streams"></a>
### The Streams class

The `Streams` subclass of `TopologyNode` adds support for Kafka and [fault tolerance](checkpoints.ipynb) through checkpointing the in-memory state of pydbsp to any "storage" supported by Kafi Streams, i.e., currently real Kafka or emulated Kafka on disk, S3 or Azure Blob Storage.

`Streams` continuously consumes the source topics, pushes the data to pydbsp, gets the outputs and produces them to the sink topics.


---
<a id="relationship"></a>
## The relationship between TopologyNode and Streams

What is the relationship between the two main classes of Kafi Streams, `TopologyNode` and `Streams`?

The [Quickstart](quickstart.ipynb) was based on the `Streams` subclass of the `TopologyNode` class because the aim was to give you the full picture from the start.

`Streams`, as already alluded to above, is just about adding support for Kafka to `TopologyNode`. The actual stream processing in Kafi Streams is completely independent of Kafka - it could, in principle, be fed by any source and emit the output to any sink.

Here is a practical example - the code from the example in [Quickstart](quickstart.ipynb), but based on `TopologyNode` instead of `Streams`:

In [2]:
# 1. Boilerplate

!pip install -r requirements.txt

import sys
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

# 2. Specify the Topology

click_source_str = "clicks"
customer_source_str = "customers"

## a) Clicks

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

## b) Customers

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

## c) Join and Sink

joined_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
)

# 3. Build the Topology

built_tn = Tn.build(joined_tn)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


As you can see, the topology is defined in an almost identical way. The only differences are:
* The connection to Kafka is left out, also in the source specification.
* The sources are specified using `Tn.source()` instead of `Streams.source()`.
* The sink is not explicitly specified (only possible with `TopologyNode`, not `Streams`).
* The build step is done using `Tn.build()` instead of `Streams.build()`.

Now how can supply data to the sources without Kafka? And how can we get the outputs of the processing?

Here is how we can use Kafi Streams without Kafka by using the `built_tn` object directly:

In [3]:
sink_m_list = []
for i in range(100):
    # 1. Generate new data.
    click_m_list = click_generator.generate(100)
    customer_m_list = customer_generator.generate(100)

    # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
    m_list = built_tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})

    # 3. Add the changes to the output list.
    sink_m_list += m_list

print(len(sink_m_list))
print(sink_m_list[-10:])


9023
[{'value': {'customer_id': 89, 'view_time': 82, 'ts': 1786790639294, 'name': 'Craig Hernandez'}}, {'value': {'customer_id': 89, 'view_time': 94, 'ts': 1786790636294, 'name': 'Craig Hernandez'}}, {'value': {'customer_id': 89, 'view_time': 114, 'ts': 1786790639794, 'name': 'Craig Hernandez'}}, {'value': {'customer_id': 90, 'view_time': 118, 'ts': 1786790643694, 'name': 'Sara Gutierrez'}}, {'value': {'customer_id': 90, 'view_time': 94, 'ts': 1786790643994, 'name': 'Sara Gutierrez'}}, {'value': {'customer_id': 92, 'view_time': 55, 'ts': 1786790635794, 'name': 'Ashley Patrick'}}, {'value': {'customer_id': 93, 'view_time': 27, 'ts': 1786790642494, 'name': 'Jennifer Velasquez'}}, {'value': {'customer_id': 94, 'view_time': 27, 'ts': 1786790637794, 'name': 'Deborah Le'}}, {'value': {'customer_id': 95, 'view_time': 77, 'ts': 1786790639394, 'name': 'Kevin Chen'}}, {'value': {'customer_id': 98, 'view_time': 66, 'ts': 1786790642894, 'name': 'Karina Flores'}}]


In this loop, we do the following:
1. We generate new data (10.000 clicks + 10.000 customers)
2. We push the new data to the topology using the `process()` method of the `TopologyNode` class. `process()` then processes the new data and returns the resulting changes.
3. We add the changes to the output list.

That is exactly what the `Streams` subclass does, just with Kafka for sources and sinks - and with fault tolerance implemented by checkpointing.
